In [ ]:
import sys

print(sys.executable)

In [ ]:
from ultralytics import YOLO
import torch
import cv2
import yaml

print("YOLO :", __import__("ultralytics").__version__)
print("PyTorch :", torch.__version__)
print("OpenCV :", cv2.__version__)
print("PyYAML :", yaml.__version__)

In [ ]:
from ultralytics import YOLO
import torch
import cv2
import yaml

print("YOLO :", __import__("ultralytics").__version__)
print("PyTorch :", torch.__version__)
print("OpenCV :", cv2.__version__)
print("PyYAML :", yaml.__version__)

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")
DATASET_DIR = PROJECT_DIR / "dataset"

print("Project:")
print(PROJECT_DIR)

print("\nDataset:")
print(DATASET_DIR)

print("\nProject exists:", PROJECT_DIR.exists())
print("Dataset exists:", DATASET_DIR.exists())

In [ ]:
import yaml

yaml_path = DATASET_DIR / "data.yaml"

with open(yaml_path, "r", encoding="utf-8") as file:
    data_config = yaml.safe_load(file)

data_config

In [ ]:
names = data_config["names"]

print("Classes:")
for class_id, class_name in enumerate(names):
    print(f"{class_id} -> {class_name}")

In [ ]:
from pathlib import Path

train_images = DATASET_DIR / "train" / "images"
valid_images = DATASET_DIR / "valid" / "images"

train_labels = DATASET_DIR / "train" / "labels"
valid_labels = DATASET_DIR / "valid" / "labels"

print("Train images :", len(list(train_images.glob("*"))))
print("Train labels :", len(list(train_labels.glob("*.txt"))))

print("Validation images :", len(list(valid_images.glob("*"))))
print("Validation labels :", len(list(valid_labels.glob("*.txt"))))

In [ ]:
import cv2
import matplotlib.pyplot as plt


def show_yolo_annotation(image_path, label_path, class_names):

    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    height, width, _ = image.shape

    with open(label_path, "r") as file:
        lines = file.readlines()

    for line in lines:

        values = line.strip().split()

        if len(values) != 5:
            continue

        class_id = int(values[0])

        x_center = float(values[1]) * width
        y_center = float(values[2]) * height
        box_width = float(values[3]) * width
        box_height = float(values[4]) * height

        x1 = int(x_center - box_width / 2)
        y1 = int(y_center - box_height / 2)

        x2 = int(x_center + box_width / 2)
        y2 = int(y_center + box_height / 2)

        cv2.rectangle(
            image,
            (x1, y1),
            (x2, y2),
            (255, 0, 0),
            2
        )

        label = class_names[class_id]

        cv2.putText(
            image,
            label,
            (x1, max(y1 - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 0, 0),
            2
        )

    plt.figure(figsize=(12, 8))
    plt.imshow(image)
    plt.axis("off")
    plt.show()

In [ ]:
train_files = list(train_images.glob("*"))

image_path = train_files[0]
label_path = train_labels / f"{image_path.stem}.txt"

print("Image :", image_path.name)
print("Label :", label_path.name)

show_yolo_annotation(
    image_path,
    label_path,
    names
)

In [ ]:
from collections import Counter

class_counts = Counter()

for label_file in train_labels.glob("*.txt"):
    with open(label_file, "r") as file:
        for line in file:
            values = line.strip().split()

            if len(values) == 5:
                class_id = int(values[0])
                class_counts[class_id] += 1

print("Nombre d'objets annotés :")

for class_id, count in sorted(class_counts.items()):
    print(f"{class_id} -> {names[class_id]} : {count}")

In [ ]:
from collections import Counter

images_with_class = Counter()

for label_file in train_labels.glob("*.txt"):

    classes_in_image = set()

    with open(label_file, "r") as file:
        for line in file:

            values = line.strip().split()

            if len(values) == 5:
                class_id = int(values[0])
                classes_in_image.add(class_id)

    for class_id in classes_in_image:
        images_with_class[class_id] += 1

print("Images contenant chaque classe :")

for class_id, count in sorted(images_with_class.items()):
    print(f"{class_id} -> {names[class_id]} : {count} images")

In [ ]:
print(data_config)

In [ ]:
import cv2
import matplotlib.pyplot as plt

image_path = train_files[0]

image = cv2.imread(str(image_path))
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 8))
plt.imshow(image)
plt.axis("off")
plt.title(image_path.name)
plt.show()

In [ ]:
import random

samples = random.sample(
    train_files,
    min(9, len(train_files))
)

for image_path in samples:

    label_path = train_labels / f"{image_path.stem}.txt"

    show_yolo_annotation(
        image_path,
        label_path,
        names
    )

In [ ]:
print(data_config)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

print("YOLOv8 chargé avec succès !")

In [ ]:
from ultralytics import YOLO

# Charger le modèle pré-entraîné
model = YOLO("yolov8n.pt")

# Lancer l'entraînement
results = model.train(
    data=str(DATASET_DIR / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    project=str(PROJECT_DIR / "runs"),
    name="safe_driving"
)

print("Training completed!")

In [ ]:
from ultralytics import YOLO
from pathlib import Path

MODEL_PATH = PROJECT_DIR / "runs" / "safe_driving-3" / "weights" / "best.pt"

model = YOLO(str(MODEL_PATH))

print("Modèle chargé :")
print(MODEL_PATH)

In [ ]:
print(model.names)

In [ ]:
# Sélectionner une image de validation
val_images_dir = DATASET_DIR / "valid" / "images"

image_path = list(val_images_dir.glob("*"))[0]

print("Image testée :", image_path)

In [ ]:
# Faire une prédiction
results = model.predict(
    source=str(image_path),
    conf=0.25,
    save=False
)

print("Prédiction terminée.")

In [ ]:
import matplotlib.pyplot as plt

result = results[0]

annotated_image = result.plot()

plt.figure(figsize=(12, 8))
plt.imshow(annotated_image[:, :, ::-1])
plt.axis("off")
plt.show()

In [ ]:
image_path = list(val_images_dir.glob("*"))[0]

In [ ]:
from pathlib import Path

val_images_dir = DATASET_DIR / "valid" / "images"
val_labels_dir = DATASET_DIR / "valid" / "labels"

seatbelt_image = None

for label_file in val_labels_dir.glob("*.txt"):
    with open(label_file, "r") as f:
        lines = f.readlines()

    # Classe 1 = seatbelt
    if any(line.strip().split()[0] == "1" for line in lines if line.strip()):
        image_name = label_file.stem
        possible_images = list(val_images_dir.glob(image_name + ".*"))

        if possible_images:
            seatbelt_image = possible_images[0]
            break

print("Image trouvée :", seatbelt_image)

In [ ]:
results = model.predict(
    source=str(seatbelt_image),
    conf=0.25,
    save=False
)

result = results[0]

print("Classes détectées :", result.names)
print("Nombre de détections :", len(result.boxes))

In [ ]:
import matplotlib.pyplot as plt

annotated_image = result.plot()

plt.figure(figsize=(12, 8))
plt.imshow(annotated_image[:, :, ::-1])
plt.axis("off")
plt.show()

In [ ]:
print("Image :", seatbelt_image)

label_file = val_labels_dir / (seatbelt_image.stem + ".txt")

print("\nAnnotations :")
with open(label_file, "r") as f:
    for line in f:
        print(line.strip())

In [ ]:
show_yolo_annotation(
    seatbelt_image,
    label_file,
    model.names
)

In [ ]:
results = model.predict(
    source=str(seatbelt_image),
    conf=0.01,
    save=False
)

result = results[0]

print("Détections :", len(result.boxes))

for box in result.boxes:
    class_id = int(box.cls[0])
    confidence = float(box.conf[0])

    print(
        f"{model.names[class_id]} → {confidence:.3f}"
    )

In [ ]:
for box in result.boxes:
    class_id = int(box.cls[0])
    confidence = float(box.conf[0])

    print(f"{model.names[class_id]} → {confidence:.3f}")

In [ ]:
metrics = model.val(
    data=str(DATASET_DIR / "data.yaml"),
    split="val",
    imgsz=640,
    batch=16
)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

val_images = list((DATASET_DIR / "valid" / "images").glob("*"))

# Choisir une image
image_path = val_images[10]

print("Image :", image_path)

In [ ]:
results = model.predict(
    source=str(image_path),
    conf=0.25,
    save=False
)

result = results[0]

for box in result.boxes:
    class_id = int(box.cls[0])
    confidence = float(box.conf[0])
    print(f"{model.names[class_id]} → {confidence:.3f}")

In [ ]:
annotated_image = result.plot()

plt.figure(figsize=(12, 8))
plt.imshow(annotated_image[:, :, ::-1])
plt.axis("off")
plt.show()

In [ ]:
import random
import matplotlib.pyplot as plt

images_to_test = random.sample(val_images, min(6, len(val_images)))

for image_path in images_to_test:

    results = model.predict(
        source=str(image_path),
        conf=0.25,
        save=False,
        verbose=False
    )

    result = results[0]

    print(f"\n{image_path.name}")

    for box in result.boxes:
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])
        print(f"  {model.names[class_id]} → {confidence:.3f}")

    annotated_image = result.plot()

    plt.figure(figsize=(10, 7))
    plt.imshow(annotated_image[:, :, ::-1])
    plt.title(image_path.name)
    plt.axis("off")
    plt.show()

In [ ]:
from pathlib import Path

VIDEO_PATH = PROJECT_DIR / "video_test.mp4"

print("Vidéo :", VIDEO_PATH)

In [ ]:
print("Existe :", VIDEO_PATH.exists())

In [ ]:
results = model.predict(
    source=str(VIDEO_PATH),
    conf=0.25,
    save=True
)

In [ ]:
results = model.predict(
    source=str(VIDEO_PATH),
    conf=0.05,
    save=True
)

In [ ]:
from pathlib import Path
from ultralytics import YOLO

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")
DATASET_DIR = PROJECT_DIR / "dataset"

MODEL_PATH = PROJECT_DIR / "runs" / "safe_driving-3" / "weights" / "best.pt"

print(MODEL_PATH.exists())

In [ ]:
model = YOLO(str(MODEL_PATH))

print(model.names)

In [ ]:
VIDEO_PATH = PROJECT_DIR / "video_test.mp4"

results = model.predict(
    source=str(VIDEO_PATH),
    conf=0.05,
    save=True
)

In [ ]:
import cv2
from pathlib import Path
from ultralytics import YOLO

# =========================
# 1. Chemins
# =========================

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")

MODEL_PATH = PROJECT_DIR / "runs" / "safe_driving-3" / "weights" / "best.pt"
VIDEO_PATH = PROJECT_DIR / "video_test.mp4"

OUTPUT_PATH = PROJECT_DIR / "safe_driving_result.mp4"


# =========================
# 2. Charger le modèle
# =========================

model = YOLO(str(MODEL_PATH))

print("Modèle chargé :", MODEL_PATH)
print("Classes :", model.names)


# =========================
# 3. Ouvrir la vidéo
# =========================

cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    raise Exception("Impossible d'ouvrir la vidéo.")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("FPS :", fps)
print("Résolution :", width, "x", height)
print("Nombre de frames :", total_frames)


# =========================
# 4. Préparer la vidéo de sortie
# =========================

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    str(OUTPUT_PATH),
    fourcc,
    fps,
    (width, height)
)


# =========================
# 5. Analyse frame par frame
# =========================

frame_number = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1

    # Détection YOLO
    results = model.predict(
        source=frame,
        conf=0.05,
        verbose=False
    )

    result = results[0]

    # Dessiner les détections
    annotated_frame = result.plot()

    # Afficher la progression
    cv2.putText(
        annotated_frame,
        f"Frame: {frame_number}/{total_frames}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )

    # Sauvegarder la frame
    out.write(annotated_frame)


# =========================
# 6. Fermer les fichiers
# =========================

cap.release()
out.release()

print("Analyse terminée.")
print("Vidéo enregistrée ici :")
print(OUTPUT_PATH)

In [ ]:
results = model.predict(
    source=str(VIDEO_PATH),
    conf=0.05,
    verbose=False
)

result = results[0]

print(result.boxes)

In [ ]:
cap = cv2.VideoCapture(str(VIDEO_PATH))

ret, frame = cap.read()

cap.release()

print("Frame récupérée :", ret)

In [ ]:
results = model.predict(
    source=frame,
    conf=0.05,
    verbose=False
)

result = results[0]

print(result.boxes)

In [ ]:
for box, cls, conf in zip(
    result.boxes.xyxy,
    result.boxes.cls,
    result.boxes.conf
):

    class_id = int(cls)
    confidence = float(conf)

    class_name = model.names[class_id]

    print(
        f"{class_name} → {confidence:.2f} → {box.tolist()}"
    )

In [ ]:
cap = cv2.VideoCapture(str(VIDEO_PATH))

ret, frame = cap.read()
cap.release()

print("Frame récupérée :", ret)

results = model.predict(
    source=frame,
    conf=0.05,
    verbose=False
)

result = results[0]

for box, cls, conf in zip(
    result.boxes.xyxy,
    result.boxes.cls,
    result.boxes.conf
):
    class_id = int(cls)
    confidence = float(conf)
    class_name = model.names[class_id]

    if class_name == "seatbelt":
        print(
            f"Seatbelt détectée | "
            f"Confiance : {confidence:.2f} | "
            f"Box : {box.tolist()}"
        )

In [ ]:
SEATBELT_THRESHOLD = 0.20

In [ ]:
if class_name == "seatbelt":
    if confidence >= SEATBELT_THRESHOLD:
        print("✅ Seatbelt détectée")
    else:
        print("⚠️ Détection trop faible")

In [ ]:
import cv2
from pathlib import Path
from ultralytics import YOLO
from collections import deque


# ============================================================
# 1. CHEMINS
# ============================================================

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")

MODEL_PATH = (
    PROJECT_DIR
    / "runs"
    / "safe_driving-3"
    / "weights"
    / "best.pt"
)

VIDEO_PATH = PROJECT_DIR / "video_test.mp4"

OUTPUT_PATH = PROJECT_DIR / "safe_driving_final.mp4"


# ============================================================
# 2. CHARGEMENT DU MODÈLE
# ============================================================

model = YOLO(str(MODEL_PATH))

print("Modèle chargé :", MODEL_PATH)
print("Classes :", model.names)


# ============================================================
# 3. PARAMÈTRES
# ============================================================

YOLO_CONFIDENCE = 0.05

SEATBELT_THRESHOLD = 0.15
MOBILE_THRESHOLD = 0.20

WINDOW_SIZE = 10
MIN_POSITIVE_FRAMES = 6


# ============================================================
# 4. FONCTION DE DÉCISION
# ============================================================

def make_decision(seatbelt_confirmed, mobile_detected):

    if mobile_detected:
        return "WARNING - MOBILE DETECTED"

    elif seatbelt_confirmed:
        return "SAFE - SEATBELT CONFIRMED"

    else:
        return "CHECKING SEATBELT"


# ============================================================
# 5. OUVERTURE DE LA VIDÉO
# ============================================================

cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    raise Exception("Impossible d'ouvrir la vidéo.")


fps = cap.get(cv2.CAP_PROP_FPS)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))


print()
print("===== INFORMATIONS VIDÉO =====")
print("FPS :", fps)
print("Largeur :", width)
print("Hauteur :", height)
print("Frames :", total_frames)


# ============================================================
# 6. VIDÉO DE SORTIE
# ============================================================

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    str(OUTPUT_PATH),
    fourcc,
    fps,
    (width, height)
)


# ============================================================
# 7. FENÊTRE GLISSANTE
# ============================================================

seatbelt_history = deque(maxlen=WINDOW_SIZE)


# ============================================================
# 8. TRAITEMENT
# ============================================================

frame_number = 0


while True:

    # --------------------------------------------------------
    # Lire une frame
    # --------------------------------------------------------

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1


    # --------------------------------------------------------
    # YOLO
    # --------------------------------------------------------

    results = model.predict(
        source=frame,
        conf=YOLO_CONFIDENCE,
        verbose=False
    )

    result = results[0]


    # --------------------------------------------------------
    # Variables de détection
    # --------------------------------------------------------

    seatbelt_detected = False
    mobile_detected = False


    # ========================================================
    # ANALYSE DES BOXES
    # ========================================================

    for box, cls, conf in zip(
        result.boxes.xyxy,
        result.boxes.cls,
        result.boxes.conf
    ):

        class_id = int(cls)

        confidence = float(conf)

        class_name = model.names[class_id]


        # ----------------------------------------------------
        # SEATBELT
        # ----------------------------------------------------

        if (
            class_name == "seatbelt"
            and confidence >= SEATBELT_THRESHOLD
        ):
            seatbelt_detected = True


        # ----------------------------------------------------
        # MOBILE
        # ----------------------------------------------------

        if (
            class_name == "mobile"
            and confidence >= MOBILE_THRESHOLD
        ):
            mobile_detected = True


    # ========================================================
    # TEMPORAL SMOOTHING
    # ========================================================

    seatbelt_history.append(seatbelt_detected)

    positive_frames = sum(seatbelt_history)


    # --------------------------------------------------------
    # Confirmation de la ceinture
    # --------------------------------------------------------

    seatbelt_confirmed = (
        positive_frames >= MIN_POSITIVE_FRAMES
    )


    # ========================================================
    # DÉCISION
    # ========================================================

    status = make_decision(
        seatbelt_confirmed,
        mobile_detected
    )


    # ========================================================
    # AFFICHAGE YOLO
    # ========================================================

    annotated_frame = result.plot()


    # ========================================================
    # AFFICHAGE DU STATUT
    # ========================================================

    cv2.putText(
        annotated_frame,
        status,
        (20, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 255, 255),
        2
    )


    # ========================================================
    # INFORMATIONS CEINTURE
    # ========================================================

    cv2.putText(
        annotated_frame,
        f"Seatbelt: "
        f"{positive_frames}/{len(seatbelt_history)}",
        (20, 90),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )


    # ========================================================
    # INFORMATIONS MOBILE
    # ========================================================

    cv2.putText(
        annotated_frame,
        f"Mobile: "
        f"{'DETECTED' if mobile_detected else 'NOT DETECTED'}",
        (20, 130),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )


    # ========================================================
    # NUMÉRO FRAME
    # ========================================================

    cv2.putText(
        annotated_frame,
        f"Frame: {frame_number}/{total_frames}",
        (20, 170),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )


    # ========================================================
    # ÉCRITURE
    # ========================================================

    out.write(annotated_frame)


# ============================================================
# 9. FERMETURE
# ============================================================

cap.release()
out.release()


print()
print("========================================")
print("Analyse terminée.")
print("Vidéo générée :")
print(OUTPUT_PATH)
print("========================================")

In [ ]:
from pathlib import Path

# ============================================================
# CHEMINS
# ============================================================

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")

IMAGES_DIR = PROJECT_DIR / "dataset" / "train" / "images"
LABELS_DIR = PROJECT_DIR / "dataset" / "train" / "labels"

# Classes actuelles
CLASS_NAMES = {
    0: "mobile",
    1: "seatbelt",
    2: "windshield"
}


# ============================================================
# ANALYSE
# ============================================================

image_files = list(IMAGES_DIR.glob("*"))

total_images = 0
images_with_seatbelt = []
images_without_seatbelt = []

for image_path in image_files:

    # On garde uniquement les images
    if image_path.suffix.lower() not in [
        ".jpg", ".jpeg", ".png", ".bmp", ".webp"
    ]:
        continue

    total_images += 1

    # Label correspondant
    label_path = LABELS_DIR / f"{image_path.stem}.txt"

    has_seatbelt = False

    if label_path.exists():

        with open(label_path, "r") as file:

            for line in file:

                values = line.strip().split()

                if len(values) != 5:
                    continue

                class_id = int(values[0])

                if class_id == 1:
                    has_seatbelt = True
                    break

    if has_seatbelt:
        images_with_seatbelt.append(image_path)

    else:
        images_without_seatbelt.append(image_path)


# ============================================================
# RÉSULTATS
# ============================================================

print("==========================================")
print("ANALYSE DU DATASET")
print("==========================================")

print(f"Nombre total d'images       : {total_images}")
print(f"Avec seatbelt               : {len(images_with_seatbelt)}")
print(f"Sans annotation seatbelt    : {len(images_without_seatbelt)}")

if total_images > 0:

    percentage = (
        len(images_without_seatbelt)
        / total_images
        * 100
    )

    print(
        f"Pourcentage sans seatbelt  : "
        f"{percentage:.2f}%"
    )

In [ ]:
import matplotlib.pyplot as plt
import cv2
import math


# ============================================================
# NOMBRE D'IMAGES À AFFICHER
# ============================================================

N = 20

# Prendre les 20 premières images
samples = images_without_seatbelt[:N]


# ============================================================
# AFFICHAGE
# ============================================================

cols = 4
rows = math.ceil(len(samples) / cols)

plt.figure(figsize=(16, 4 * rows))


for i, image_path in enumerate(samples):

    # Lire l'image
    image = cv2.imread(str(image_path))

    # OpenCV → RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Créer le subplot
    plt.subplot(rows, cols, i + 1)

    # Afficher
    plt.imshow(image)

    # Nom de l'image
    plt.title(image_path.name, fontsize=9)

    plt.axis("off")


plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import cv2
import math

# ============================================================
# PARAMÈTRES
# ============================================================

BATCH_SIZE = 20
batch_number = 0

total_batches = math.ceil(
    len(images_without_seatbelt) / BATCH_SIZE
)

print(f"Nombre total de lots : {total_batches}")

In [ ]:
# ============================================================
# AFFICHER UN LOT
# ============================================================

def show_batch(batch_number):

    start = batch_number * BATCH_SIZE
    end = start + BATCH_SIZE

    samples = images_without_seatbelt[start:end]

    if not samples:
        print("Ce lot n'existe pas.")
        return

    cols = 4
    rows = math.ceil(len(samples) / cols)

    plt.figure(figsize=(16, 4 * rows))

    for i, image_path in enumerate(samples):

        image = cv2.imread(str(image_path))

        if image is None:
            continue

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        plt.subplot(rows, cols, i + 1)
        plt.imshow(image)
        plt.title(
            f"{start + i + 1}. {image_path.name}",
            fontsize=8
        )
        plt.axis("off")

    plt.tight_layout()
    plt.show()

    print(
        f"Images {start + 1} à "
        f"{min(end, len(images_without_seatbelt))} "
        f"/ {len(images_without_seatbelt)}"
    )

In [ ]:
show_batch(6)

In [ ]:
from pathlib import Path
import shutil

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")

NO_SEATBELT_DIR = PROJECT_DIR / "no_seatbelt_images"
NO_SEATBELT_DIR.mkdir(exist_ok=True)

print("Dossier créé :", NO_SEATBELT_DIR)

In [ ]:
import shutil
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")

NO_SEATBELT_DIR = PROJECT_DIR / "no_seatbelt_images"
NO_SEATBELT_DIR.mkdir(exist_ok=True)

# Les 300 premières images que tu as validées
selected_images = images_without_seatbelt[:300]

for image_path in selected_images:
    destination = NO_SEATBELT_DIR / image_path.name
    shutil.copy2(image_path, destination)

print(f"{len(selected_images)} images copiées.")
print(f"Dossier : {NO_SEATBELT_DIR}")

In [ ]:
print("Nombre d'images :", len(list(NO_SEATBELT_DIR.glob("*"))))

In [ ]:
from pathlib import Path
from ultralytics import YOLO

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")

MODEL_PATH = (
    PROJECT_DIR
    / "runs"
    / "safe_driving-3"
    / "weights"
    / "best.pt"
)

NO_SEATBELT_DIR = PROJECT_DIR / "no_seatbelt_images"

model = YOLO(str(MODEL_PATH))

print("Modèle :", MODEL_PATH)
print("Images :", NO_SEATBELT_DIR)
print("Nombre d'images :", len(list(NO_SEATBELT_DIR.glob("*"))))
print("Classes :", model.names)

In [ ]:
seatbelt_false_positives = []

SEATBELT_THRESHOLD = 0.20

image_files = [
    p for p in NO_SEATBELT_DIR.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
]

for image_path in image_files:

    results = model.predict(
        source=str(image_path),
        conf=0.05,
        verbose=False
    )

    result = results[0]

    max_seatbelt_conf = 0.0

    for cls, conf in zip(
        result.boxes.cls,
        result.boxes.conf
    ):
        class_id = int(cls)
        confidence = float(conf)

        if class_id == 1:  # seatbelt
            max_seatbelt_conf = max(
                max_seatbelt_conf,
                confidence
            )

    if max_seatbelt_conf >= SEATBELT_THRESHOLD:
        seatbelt_false_positives.append(
            (image_path, max_seatbelt_conf)
        )

print("===================================")
print("TEST DES 300 IMAGES")
print("===================================")
print("Images testées :", len(image_files))
print("Fausses détections seatbelt :", len(seatbelt_false_positives))

false_positive_rate = (
    len(seatbelt_false_positives) / len(image_files) * 100
)

print(f"Taux de faux positifs : {false_positive_rate:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
import cv2
import math

N = len(seatbelt_false_positives)

cols = 3
rows = math.ceil(N / cols)

plt.figure(figsize=(15, 5 * rows))

for i, (image_path, confidence) in enumerate(seatbelt_false_positives):

    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    plt.subplot(rows, cols, i + 1)
    plt.imshow(image)

    plt.title(
        f"{image_path.name}\n"
        f"Seatbelt confidence = {confidence:.3f}",
        fontsize=10
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]

print("=" * 55)
print("FAUX POSITIFS SELON LE SEUIL")
print("=" * 55)

for threshold in thresholds:

    false_positives = 0

    for image_path in image_files:

        results = model.predict(
            source=str(image_path),
            conf=0.05,
            verbose=False
        )

        result = results[0]

        max_seatbelt_conf = 0.0

        for cls, conf in zip(
            result.boxes.cls,
            result.boxes.conf
        ):
            class_id = int(cls)
            confidence = float(conf)

            if class_id == 1:  # seatbelt
                max_seatbelt_conf = max(
                    max_seatbelt_conf,
                    confidence
                )

        if max_seatbelt_conf >= threshold:
            false_positives += 1

    rate = false_positives / len(image_files) * 100

    print(
        f"Seuil {threshold:.2f} → "
        f"{false_positives:3d}/300 "
        f"({rate:.2f} %)"
    )

In [ ]:
import cv2
import numpy as np
from pathlib import Path

# =========================
# PARAMÈTRES
# =========================

VAL_IMAGES_DIR = DATASET_DIR / "valid" / "images"
VAL_LABELS_DIR = DATASET_DIR / "valid" / "labels"

IOU_THRESHOLD = 0.50
YOLO_CONF = 0.05

THRESHOLDS = [
    0.10, 0.15, 0.20, 0.25,
    0.30, 0.40, 0.50, 0.60,
    0.70, 0.80
]

# =========================
# FONCTION IoU
# =========================

def calculate_iou(box1, box2):
    """
    box = [x1, y1, x2, y2]
    """

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection_width = max(0, x2 - x1)
    intersection_height = max(0, y2 - y1)

    intersection = intersection_width * intersection_height

    area1 = (
        max(0, box1[2] - box1[0])
        * max(0, box1[3] - box1[1])
    )

    area2 = (
        max(0, box2[2] - box2[0])
        * max(0, box2[3] - box2[1])
    )

    union = area1 + area2 - intersection

    if union == 0:
        return 0.0

    return intersection / union


# =========================
# RÉCUPÉRER LES IMAGES
# =========================

val_images = [
    p for p in VAL_IMAGES_DIR.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
]

print("Images de validation :", len(val_images))


# =========================
# PRÉPARER LES DONNÉES
# =========================

all_predictions = []
total_ground_truth = 0

for image_path in val_images:

    image = cv2.imread(str(image_path))

    if image is None:
        continue

    height, width = image.shape[:2]

    # -------------------------
    # Ground truth seatbelt
    # -------------------------

    label_path = VAL_LABELS_DIR / f"{image_path.stem}.txt"

    gt_boxes = []

    if label_path.exists():

        with open(label_path, "r") as f:

            for line in f:

                values = line.strip().split()

                if len(values) != 5:
                    continue

                class_id = int(values[0])

                # Seulement seatbelt
                if class_id != 1:
                    continue

                x_center = float(values[1]) * width
                y_center = float(values[2]) * height
                box_width = float(values[3]) * width
                box_height = float(values[4]) * height

                x1 = x_center - box_width / 2
                y1 = y_center - box_height / 2
                x2 = x_center + box_width / 2
                y2 = y_center + box_height / 2

                gt_boxes.append(
                    [x1, y1, x2, y2]
                )

    total_ground_truth += len(gt_boxes)

    # -------------------------
    # Prédictions YOLO
    # -------------------------

    results = model.predict(
        source=str(image_path),
        conf=YOLO_CONF,
        verbose=False
    )

    result = results[0]

    predictions = []

    for box, cls, conf in zip(
        result.boxes.xyxy,
        result.boxes.cls,
        result.boxes.conf
    ):

        class_id = int(cls)

        if class_id != 1:
            continue

        predictions.append({
            "box": box.tolist(),
            "confidence": float(conf)
        })

    all_predictions.append({
        "image": image_path,
        "ground_truth": gt_boxes,
        "predictions": predictions
    })


print("Nombre total de ground-truth seatbelt :", total_ground_truth)
print("Préparation terminée.")

In [ ]:
results_by_threshold = []

for threshold in THRESHOLDS:

    TP = 0
    FP = 0
    FN = 0

    for item in all_predictions:

        gt_boxes = item["ground_truth"]

        predictions = [
            p for p in item["predictions"]
            if p["confidence"] >= threshold
        ]

        matched_gt = set()

        # Trier les prédictions par confiance
        predictions = sorted(
            predictions,
            key=lambda x: x["confidence"],
            reverse=True
        )

        for prediction in predictions:

            best_iou = 0
            best_gt_index = None

            for gt_index, gt_box in enumerate(gt_boxes):

                if gt_index in matched_gt:
                    continue

                iou = calculate_iou(
                    prediction["box"],
                    gt_box
                )

                if iou > best_iou:
                    best_iou = iou
                    best_gt_index = gt_index

            if (
                best_gt_index is not None
                and best_iou >= IOU_THRESHOLD
            ):
                TP += 1
                matched_gt.add(best_gt_index)

            else:
                FP += 1

        FN += len(gt_boxes) - len(matched_gt)

    precision = (
        TP / (TP + FP)
        if (TP + FP) > 0
        else 0
    )

    recall = (
        TP / (TP + FN)
        if (TP + FN) > 0
        else 0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    results_by_threshold.append({
        "threshold": threshold,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })


# =========================
# AFFICHAGE
# =========================

print("=" * 80)
print("ÉVALUATION SEATBELT — VALIDATION SET")
print("=" * 80)

print(
    f"{'Seuil':<10}"
    f"{'TP':<8}"
    f"{'FP':<8}"
    f"{'FN':<8}"
    f"{'Precision':<12}"
    f"{'Recall':<12}"
    f"{'F1':<10}"
)

print("-" * 80)

for r in results_by_threshold:

    print(
        f"{r['threshold']:<10.2f}"
        f"{r['TP']:<8}"
        f"{r['FP']:<8}"
        f"{r['FN']:<8}"
        f"{r['precision']*100:<11.2f}%"
        f"{r['recall']*100:<11.2f}%"
        f"{r['f1']:.3f}"
    )

In [ ]:
THRESHOLD = 0.25

image_level_results = []

for item in all_predictions:

    has_real_seatbelt = len(item["ground_truth"]) > 0

    detected_seatbelt = any(
        p["confidence"] >= THRESHOLD
        for p in item["predictions"]
    )

    image_level_results.append({
        "image": item["image"],
        "real_seatbelt": has_real_seatbelt,
        "detected_seatbelt": detected_seatbelt
    })


# =========================
# COMPTAGE
# =========================

true_positive_images = 0
false_negative_images = 0
false_positive_images = 0
true_negative_images = 0

for item in image_level_results:

    real = item["real_seatbelt"]
    detected = item["detected_seatbelt"]

    if real and detected:
        true_positive_images += 1

    elif real and not detected:
        false_negative_images += 1

    elif not real and detected:
        false_positive_images += 1

    elif not real and not detected:
        true_negative_images += 1


print("=" * 60)
print("ÉVALUATION AU NIVEAU IMAGE")
print("=" * 60)

print("Images avec vraie seatbelt :", 
      true_positive_images + false_negative_images)

print("Images sans seatbelt :", 
      false_positive_images + true_negative_images)

print()

print("Vraies détections :", true_positive_images)
print("Seatbelt manquées :", false_negative_images)
print("Fausses détections :", false_positive_images)
print("Rejets corrects :", true_negative_images)

In [ ]:
total_positive = (
    true_positive_images +
    false_negative_images
)

total_negative = (
    false_positive_images +
    true_negative_images
)

image_recall = (
    true_positive_images / total_positive
    if total_positive > 0 else 0
)

false_positive_rate = (
    false_positive_images / total_negative
    if total_negative > 0 else 0
)

print()
print(f"Recall au niveau image : {image_recall * 100:.2f}%")
print(f"Taux de faux positifs : {false_positive_rate * 100:.2f}%")

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")

DATASET_V2_DIR = PROJECT_DIR / "dataset_v2"
NO_SEATBELT_DIR = PROJECT_DIR / "no_seatbelt_images"

print("Dataset V2 :", DATASET_V2_DIR.exists())
print("Images no_seatbelt :", NO_SEATBELT_DIR.exists())

# Compter les images
images = [
    p for p in NO_SEATBELT_DIR.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
]

print("Nombre d'images sélectionnées :", len(images))

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")

DATASET_V2_DIR = PROJECT_DIR / "dataset_v2"
NO_SEATBELT_DIR = PROJECT_DIR / "no_seatbelt_images"

# Dossiers des annotations originales
ORIGINAL_IMAGES_DIR = DATASET_V2_DIR / "train" / "images"
ORIGINAL_LABELS_DIR = DATASET_V2_DIR / "train" / "labels"

# Prendre quelques images pour inspection
images = [
    p for p in NO_SEATBELT_DIR.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
]

print("Nombre d'images :", len(images))
print("\nExemple d'annotations :\n")

for image_path in images[:10]:
    label_path = ORIGINAL_LABELS_DIR / f"{image_path.stem}.txt"

    print(f"Image : {image_path.name}")

    if label_path.exists():
        with open(label_path, "r") as f:
            content = f.read().strip()

        print("Label :")
        print(content if content else "(fichier vide)")
    else:
        print("Aucun fichier label trouvé.")

    print("-" * 50)

In [ ]:
from pathlib import Path
import yaml

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")

# ⚠️ Mets ici le nom exact du dossier que tu as extrait
DATASET_V2_ROBOFLOW = PROJECT_DIR / "dataset_v2_roboflow"

DATA_YAML = DATASET_V2_ROBOFLOW / "data.yaml"

print("Dataset trouvé :", DATASET_V2_ROBOFLOW.exists())
print("data.yaml trouvé :", DATA_YAML.exists())

# Lire data.yaml
with open(DATA_YAML, "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)

print("\n=== CLASSES ===")
print("Nombre de classes :", data["nc"])
print("Classes :", data["names"])

print("\n=== STRUCTURE ===")
for folder in [
    DATASET_V2_ROBOFLOW / "train" / "images",
    DATASET_V2_ROBOFLOW / "train" / "labels",
    DATASET_V2_ROBOFLOW / "valid" / "images",
    DATASET_V2_ROBOFLOW / "valid" / "labels",
    DATASET_V2_ROBOFLOW / "test" / "images",
    DATASET_V2_ROBOFLOW / "test" / "labels",
]:
    print(folder, "→", folder.exists())

In [ ]:
CLASS_NAMES = data["names"]

print("\n=== NOMBRE D'IMAGES ===")

for split in ["train", "valid", "test"]:
    images_dir = DATASET_V2_ROBOFLOW / split / "images"
    labels_dir = DATASET_V2_ROBOFLOW / split / "labels"

    images = [
        p for p in images_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    ]

    labels = list(labels_dir.glob("*.txt"))

    print(f"{split}: {len(images)} images | {len(labels)} labels")


print("\n=== ANNOTATIONS PAR CLASSE ===")

class_counts = {i: 0 for i in range(len(CLASS_NAMES))}

for split in ["train", "valid", "test"]:
    labels_dir = DATASET_V2_ROBOFLOW / split / "labels"

    for label_file in labels_dir.glob("*.txt"):
        with open(label_file, "r", encoding="utf-8") as f:
            for line in f:
                values = line.strip().split()

                if len(values) != 5:
                    continue

                class_id = int(values[0])

                if class_id in class_counts:
                    class_counts[class_id] += 1

for class_id, count in class_counts.items():
    print(f"{class_id} - {CLASS_NAMES[class_id]} : {count}")

In [ ]:
from ultralytics import YOLO
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")
DATASET_V2_DIR = PROJECT_DIR / "dataset_v2_roboflow"

# Modèle pré-entraîné
model_v2 = YOLO("yolov8n.pt")

# Entraînement
results_v2 = model_v2.train(
    data=str(DATASET_V2_DIR / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    project=str(PROJECT_DIR / "runs"),
    name="safe_driving_v2",
    device="cpu"
)

In [ ]:
from pathlib import Path
from ultralytics import YOLO

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")
DATASET_V2_DIR = PROJECT_DIR / "dataset_v2_roboflow"

MODEL_V2_PATH = (
    PROJECT_DIR
    / "runs"
    / "safe_driving_v2"
    / "weights"
    / "best.pt"
)

print("Modèle :", MODEL_V2_PATH)
print("Existe :", MODEL_V2_PATH.exists())

model_v2 = YOLO(str(MODEL_V2_PATH))

print("Classes :", model_v2.names)

In [ ]:
test_metrics = model_v2.val(
    data=str(DATASET_V2_DIR / "data.yaml"),
    split="test",
    imgsz=640,
    batch=16,
    device="cpu"
)

In [ ]:
from pathlib import Path
from ultralytics import YOLO

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")
DATASET_V2_DIR = PROJECT_DIR / "dataset_v2_roboflow"

MODEL_V2_PATH = (
    PROJECT_DIR
    / "runs"
    / "safe_driving_v2"
    / "weights"
    / "best.pt"
)

model_v2 = YOLO(str(MODEL_V2_PATH))

metrics_test = model_v2.val(
    data=str(DATASET_V2_DIR / "data.yaml"),
    split="test",
    imgsz=640,
    batch=16,
    device="cpu",
    plots=True
)

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")

MODEL_PATH = (
    PROJECT_DIR
    / "runs"
    / "safe_driving_v2"
    / "weights"
    / "best.pt"
)

IMAGE_PATH = PROJECT_DIR / "no_seatbelt_images" / "1_BUS_16-37-51-347_jpg.rf.YCG01f7GmWZ5PMnRJcN3.jpg"

model_v2 = YOLO(str(MODEL_PATH))

results = model_v2.predict(
    source=str(IMAGE_PATH),
    conf=0.20,
    verbose=False
)

result = results[0]

annotated = result.plot()

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
from pathlib import Path
import shutil
from ultralytics import YOLO

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")

INPUT_DIR = PROJECT_DIR / "no_seatbelt_images"

OUTPUT_DIR = PROJECT_DIR / "auto_no_seatbelt"
OUTPUT_IMAGES = OUTPUT_DIR / "images"
OUTPUT_LABELS = OUTPUT_DIR / "labels"

OUTPUT_IMAGES.mkdir(parents=True, exist_ok=True)
OUTPUT_LABELS.mkdir(parents=True, exist_ok=True)

print("Images source :", INPUT_DIR)
print("Dossier sortie :", OUTPUT_DIR)

In [ ]:
images = (
    list(INPUT_DIR.glob("*.jpg"))
    + list(INPUT_DIR.glob("*.jpeg"))
    + list(INPUT_DIR.glob("*.png"))
)

print("Nombre d'images :", len(images))

In [ ]:
person_model = YOLO("yolov8n.pt")

In [ ]:
test_image = images[0]

results = person_model.predict(
    source=str(test_image),
    conf=0.30,
    classes=[0],
    verbose=False
)

result = results[0]

print("Personnes détectées :", len(result.boxes))

In [ ]:
import cv2
import matplotlib.pyplot as plt

annotated = result.plot()

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
import cv2
import shutil
from pathlib import Path
from ultralytics import YOLO

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")

INPUT_DIR = PROJECT_DIR / "no_seatbelt_images"
OUTPUT_DIR = PROJECT_DIR / "auto_no_seatbelt"

OUTPUT_IMAGES = OUTPUT_DIR / "images"
OUTPUT_LABELS = OUTPUT_DIR / "labels"

OUTPUT_IMAGES.mkdir(parents=True, exist_ok=True)
OUTPUT_LABELS.mkdir(parents=True, exist_ok=True)

# Modèle COCO
person_model = YOLO("yolov8n.pt")

# Classe V2
NO_SEATBELT_CLASS_ID = 1

# Seuil de détection personne
PERSON_CONF = 0.30

images = (
    list(INPUT_DIR.glob("*.jpg"))
    + list(INPUT_DIR.glob("*.jpeg"))
    + list(INPUT_DIR.glob("*.png"))
)

print("Images trouvées :", len(images))

generated = 0
no_person = 0
multiple_persons = 0

for image_path in images:

    # Lire image
    image = cv2.imread(str(image_path))

    if image is None:
        continue

    height, width = image.shape[:2]

    # Détection des personnes
    results = person_model.predict(
        source=image,
        conf=PERSON_CONF,
        classes=[0],
        verbose=False
    )

    result = results[0]

    person_boxes = []

    for box, conf in zip(
        result.boxes.xyxy,
        result.boxes.conf
    ):
        confidence = float(conf)

        if confidence >= PERSON_CONF:
            x1, y1, x2, y2 = box.tolist()
            person_boxes.append(
                (x1, y1, x2, y2, confidence)
            )

    if len(person_boxes) == 0:
        no_person += 1
        continue

    if len(person_boxes) > 1:
        multiple_persons += 1

    # ------------------------------------------------
    # Choisir la personne principale
    # ------------------------------------------------

    # On prend la personne avec la plus grande surface
    person_boxes.sort(
        key=lambda b: (b[2] - b[0]) * (b[3] - b[1]),
        reverse=True
    )

    x1, y1, x2, y2, confidence = person_boxes[0]

    # ------------------------------------------------
    # Zone supérieure du corps
    # ------------------------------------------------

    person_height = y2 - y1
    person_width = x2 - x1

    # On ignore la partie basse du corps
    torso_y1 = y1 + 0.15 * person_height
    torso_y2 = y1 + 0.65 * person_height

    # Réduction horizontale légère
    torso_x1 = x1 + 0.15 * person_width
    torso_x2 = x2 - 0.15 * person_width

    # Limites image
    torso_x1 = max(0, min(width - 1, torso_x1))
    torso_y1 = max(0, min(height - 1, torso_y1))
    torso_x2 = max(0, min(width - 1, torso_x2))
    torso_y2 = max(0, min(height - 1, torso_y2))

    # Vérification
    if torso_x2 <= torso_x1 or torso_y2 <= torso_y1:
        continue

    # ------------------------------------------------
    # Conversion YOLO
    # ------------------------------------------------

    x_center = ((torso_x1 + torso_x2) / 2) / width
    y_center = ((torso_y1 + torso_y2) / 2) / height

    box_width = (torso_x2 - torso_x1) / width
    box_height = (torso_y2 - torso_y1) / height

    # ------------------------------------------------
    # Sauvegarder label
    # ------------------------------------------------

    label_path = OUTPUT_LABELS / f"{image_path.stem}.txt"

    with open(label_path, "w") as f:
        f.write(
            f"{NO_SEATBELT_CLASS_ID} "
            f"{x_center:.6f} "
            f"{y_center:.6f} "
            f"{box_width:.6f} "
            f"{box_height:.6f}\n"
        )

    # Copier image
    shutil.copy2(
        image_path,
        OUTPUT_IMAGES / image_path.name
    )

    generated += 1

print()
print("========== RÉSULTAT ==========")
print("Images traitées :", len(images))
print("Annotations générées :", generated)
print("Aucune personne détectée :", no_person)
print("Plusieurs personnes détectées :", multiple_persons)

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt

auto_images = list(OUTPUT_IMAGES.glob("*.jpg"))

sample_images = random.sample(
    auto_images,
    min(10, len(auto_images))
)

for image_path in sample_images:

    image = cv2.imread(str(image_path))

    label_path = OUTPUT_LABELS / f"{image_path.stem}.txt"

    if not label_path.exists():
        continue

    height, width = image.shape[:2]

    with open(label_path, "r") as f:
        values = f.readline().strip().split()

    x_center = float(values[1]) * width
    y_center = float(values[2]) * height
    box_width = float(values[3]) * width
    box_height = float(values[4]) * height

    x1 = int(x_center - box_width / 2)
    y1 = int(y_center - box_height / 2)
    x2 = int(x_center + box_width / 2)
    y2 = int(y_center + box_height / 2)

    cv2.rectangle(
        image,
        (x1, y1),
        (x2, y2),
        (255, 0, 0),
        3
    )

    cv2.putText(
        image,
        "AUTO no_seatbelt",
        (x1, max(30, y1 - 10)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 0, 0),
        2
    )

    plt.figure(figsize=(10, 7))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(image_path.name)
    plt.axis("off")
    plt.show()

In [ ]:
# Deuxième passage avec un seuil plus bas

PERSON_CONF_V2 = 0.10

remaining_images = [
    img for img in images
    if not (OUTPUT_LABELS / f"{img.stem}.txt").exists()
]

print("Images restantes :", len(remaining_images))

generated_second_pass = 0

for image_path in remaining_images:

    image = cv2.imread(str(image_path))

    if image is None:
        continue

    height, width = image.shape[:2]

    results = person_model.predict(
        source=image,
        conf=PERSON_CONF_V2,
        classes=[0],
        verbose=False
    )

    result = results[0]

    person_boxes = []

    for box, conf in zip(
        result.boxes.xyxy,
        result.boxes.conf
    ):
        confidence = float(conf)

        if confidence >= PERSON_CONF_V2:
            x1, y1, x2, y2 = box.tolist()

            person_boxes.append(
                (x1, y1, x2, y2, confidence)
            )

    if len(person_boxes) == 0:
        continue

    # Plus grande personne
    person_boxes.sort(
        key=lambda b: (b[2] - b[0]) * (b[3] - b[1]),
        reverse=True
    )

    x1, y1, x2, y2, confidence = person_boxes[0]

    person_height = y2 - y1
    person_width = x2 - x1

    # Zone torse
    torso_y1 = y1 + 0.15 * person_height
    torso_y2 = y1 + 0.65 * person_height

    torso_x1 = x1 + 0.15 * person_width
    torso_x2 = x2 - 0.15 * person_width

    # Limites
    torso_x1 = max(0, min(width - 1, torso_x1))
    torso_y1 = max(0, min(height - 1, torso_y1))
    torso_x2 = max(0, min(width - 1, torso_x2))
    torso_y2 = max(0, min(height - 1, torso_y2))

    if torso_x2 <= torso_x1 or torso_y2 <= torso_y1:
        continue

    # YOLO format
    x_center = ((torso_x1 + torso_x2) / 2) / width
    y_center = ((torso_y1 + torso_y2) / 2) / height

    box_width = (torso_x2 - torso_x1) / width
    box_height = (torso_y2 - torso_y1) / height

    label_path = OUTPUT_LABELS / f"{image_path.stem}.txt"

    with open(label_path, "w") as f:
        f.write(
            f"1 "
            f"{x_center:.6f} "
            f"{y_center:.6f} "
            f"{box_width:.6f} "
            f"{box_height:.6f}\n"
        )

    shutil.copy2(
        image_path,
        OUTPUT_IMAGES / image_path.name
    )

    generated_second_pass += 1

print("Nouvelles annotations :", generated_second_pass)
print("Total annotations :", len(list(OUTPUT_LABELS.glob("*.txt"))))

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt

auto_images = list(OUTPUT_IMAGES.glob("*.jpg"))

sample_images = random.sample(
    auto_images,
    min(20, len(auto_images))
)

for image_path in sample_images:

    image = cv2.imread(str(image_path))

    label_path = OUTPUT_LABELS / f"{image_path.stem}.txt"

    if not label_path.exists():
        continue

    h, w = image.shape[:2]

    with open(label_path, "r") as f:
        values = f.readline().strip().split()

    x_center = float(values[1]) * w
    y_center = float(values[2]) * h
    box_width = float(values[3]) * w
    box_height = float(values[4]) * h

    x1 = int(x_center - box_width / 2)
    y1 = int(y_center - box_height / 2)
    x2 = int(x_center + box_width / 2)
    y2 = int(y_center + box_height / 2)

    cv2.rectangle(
        image,
        (x1, y1),
        (x2, y2),
        (255, 0, 0),
        3
    )

    cv2.putText(
        image,
        "no_seatbelt AUTO",
        (x1, max(30, y1 - 10)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 0, 0),
        2
    )

    plt.figure(figsize=(10, 7))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(image_path.name)
    plt.axis("off")
    plt.show()

In [ ]:
from pathlib import Path
import yaml
from collections import Counter

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")
DATASET_V3_DIR = PROJECT_DIR / "dataset_v3_roboflow"

print("Dataset trouvé :", DATASET_V3_DIR.exists())
print("data.yaml trouvé :", (DATASET_V3_DIR / "data.yaml").exists())

# Lire le YAML
with open(DATASET_V3_DIR / "data.yaml", "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)

print("\n=== CLASSES ===")
print("Nombre de classes :", data.get("nc"))
print("Classes :", data.get("names"))

print("\n=== STRUCTURE ===")
for split in ["train", "valid", "test"]:
    images_dir = DATASET_V3_DIR / split / "images"
    labels_dir = DATASET_V3_DIR / split / "labels"

    print(
        f"{split}: "
        f"{len(list(images_dir.glob('*')))} images | "
        f"{len(list(labels_dir.glob('*.txt')))} labels"
    )

print("\n=== ANNOTATIONS PAR CLASSE ===")

class_counts = Counter()

for split in ["train", "valid", "test"]:
    labels_dir = DATASET_V3_DIR / split / "labels"

    for label_file in labels_dir.glob("*.txt"):
        with open(label_file, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()

                if len(parts) >= 5:
                    class_id = int(parts[0])
                    class_counts[class_id] += 1

for class_id, class_name in enumerate(data["names"]):
    print(
        f"{class_id} - {class_name} : "
        f"{class_counts[class_id]}"
    )

In [ ]:
from ultralytics import YOLO
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")
DATASET_V3_DIR = PROJECT_DIR / "dataset_v3_roboflow"

model_v3 = YOLO("yolov8n.pt")

results_v3 = model_v3.train(
    data=str(DATASET_V3_DIR / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    project=str(PROJECT_DIR / "runs"),
    name="safe_driving_v3",
    device="cpu"
)

In [ ]:
from ultralytics import YOLO
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")
DATASET_V3_DIR = PROJECT_DIR / "dataset_v3_roboflow"

MODEL_V3_PATH = (
    PROJECT_DIR
    / "runs"
    / "safe_driving_v3"
    / "weights"
    / "best.pt"
)

model_v3 = YOLO(str(MODEL_V3_PATH))

test_metrics_v3 = model_v3.val(
    data=str(DATASET_V3_DIR / "data.yaml"),
    split="test",
    imgsz=640,
    batch=16,
    device="cpu"
)

In [ ]:
from pathlib import Path
from ultralytics import YOLO

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")

MODEL_V3_PATH = (
    PROJECT_DIR
    / "runs"
    / "safe_driving_v3"
    / "weights"
    / "best.pt"
)

DATASET_V3_DIR = PROJECT_DIR / "dataset_v3_roboflow"

model_v3 = YOLO(str(MODEL_V3_PATH))

metrics_v3 = model_v3.val(
    data=str(DATASET_V3_DIR / "data.yaml"),
    split="test",
    imgsz=640,
    batch=16,
    device="cpu",
    plots=True
)

print("Résultats :", metrics_v3.save_dir)

In [ ]:
import cv2
from pathlib import Path
from ultralytics import YOLO
from collections import deque

# =========================
# PATHS
# =========================

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")

MODEL_PATH = (
    PROJECT_DIR
    / "runs"
    / "safe_driving_v3"
    / "weights"
    / "best.pt"
)

VIDEO_PATH = PROJECT_DIR / "video_test.mp4"
OUTPUT_PATH = PROJECT_DIR / "safe_driving_v3_smoothing.mp4"

# =========================
# MODEL
# =========================

model = YOLO(str(MODEL_PATH))

# =========================
# PARAMETERS
# =========================

YOLO_CONFIDENCE = 0.20

SEATBELT_THRESHOLD = 0.25
NO_SEATBELT_THRESHOLD = 0.30
MOBILE_THRESHOLD = 0.30

WINDOW_SIZE = 10
MIN_POSITIVE_FRAMES = 6

# =========================
# VIDEO
# =========================

cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    raise Exception("Impossible d'ouvrir la vidéo.")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    str(OUTPUT_PATH),
    fourcc,
    fps,
    (width, height)
)

# =========================
# HISTORIES
# =========================

seatbelt_history = deque(maxlen=WINDOW_SIZE)
no_seatbelt_history = deque(maxlen=WINDOW_SIZE)

frame_number = 0

# =========================
# MAIN LOOP
# =========================

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1

    # YOLO detection
    results = model.predict(
        source=frame,
        conf=YOLO_CONFIDENCE,
        verbose=False
    )

    result = results[0]

    # Current frame detections
    seatbelt_detected = False
    no_seatbelt_detected = False
    mobile_detected = False

    # =========================
    # READ DETECTIONS
    # =========================

    for cls, conf in zip(
        result.boxes.cls,
        result.boxes.conf
    ):

        class_id = int(cls)
        confidence = float(conf)

        class_name = model.names[class_id]

        if (
            class_name == "seatbelt"
            and confidence >= SEATBELT_THRESHOLD
        ):
            seatbelt_detected = True

        elif (
            class_name == "no_seatbelt"
            and confidence >= NO_SEATBELT_THRESHOLD
        ):
            no_seatbelt_detected = True

        elif (
            class_name == "mobile"
            and confidence >= MOBILE_THRESHOLD
        ):
            mobile_detected = True

    # =========================
    # UPDATE HISTORIES
    # =========================

    seatbelt_history.append(seatbelt_detected)
    no_seatbelt_history.append(no_seatbelt_detected)

    seatbelt_positive = sum(seatbelt_history)
    no_seatbelt_positive = sum(no_seatbelt_history)

    # =========================
    # TEMPORAL CONFIRMATION
    # =========================

    seatbelt_confirmed = (
        seatbelt_positive >= MIN_POSITIVE_FRAMES
    )

    no_seatbelt_confirmed = (
        no_seatbelt_positive >= MIN_POSITIVE_FRAMES
    )

    # =========================
    # FINAL DECISION
    # =========================

    if mobile_detected:

        status = "WARNING - MOBILE DETECTED"

    elif (
        no_seatbelt_confirmed
        and not seatbelt_confirmed
    ):

        status = "WARNING - NO SEATBELT"

    elif seatbelt_confirmed:

        status = "SAFE - SEATBELT CONFIRMED"

    elif no_seatbelt_confirmed:

        status = "WARNING - NO SEATBELT"

    else:

        status = "CHECKING SEATBELT"

    # =========================
    # DRAW YOLO BOXES
    # =========================

    annotated_frame = result.plot()

    # Status
    cv2.putText(
        annotated_frame,
        status,
        (20, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 255, 255),
        2
    )

    # Seatbelt history
    cv2.putText(
        annotated_frame,
        f"Seatbelt: {seatbelt_positive}/{len(seatbelt_history)}",
        (20, 90),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    # No seatbelt history
    cv2.putText(
        annotated_frame,
        f"No seatbelt: {no_seatbelt_positive}/{len(no_seatbelt_history)}",
        (20, 125),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    # Frame number
    cv2.putText(
        annotated_frame,
        f"Frame: {frame_number}/{total_frames}",
        (20, 160),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    # Write frame
    out.write(annotated_frame)

# =========================
# RELEASE
# =========================

cap.release()
out.release()

print("Vidéo terminée.")
print("Résultat :", OUTPUT_PATH)

In [ ]:
import cv2
from pathlib import Path
from ultralytics import YOLO

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")
MODEL_PATH = PROJECT_DIR / "runs" / "safe_driving_v3" / "weights" / "best.pt"
VIDEO_PATH = PROJECT_DIR / "video_test.mp4"  # adapte le nom

model = YOLO(str(MODEL_PATH))

cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    raise Exception("Impossible d'ouvrir la vidéo.")

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# 5 positions dans la vidéo
positions = [
    0.00,
    0.25,
    0.50,
    0.75,
    0.95
]

for i, position in enumerate(positions):

    frame_number = int(total_frames * position)

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)

    ret, frame = cap.read()

    if not ret:
        print(f"Impossible de récupérer la frame {frame_number}")
        continue

    results = model.predict(
        source=frame,
        conf=0.05,
        verbose=False
    )

    result = results[0]

    print(f"\n========== FRAME {frame_number} ({position*100:.0f}%) ==========")

    if len(result.boxes) == 0:
        print("Aucune détection")
    else:
        for cls, conf, box in zip(
            result.boxes.cls,
            result.boxes.conf,
            result.boxes.xyxy
        ):
            class_id = int(cls)
            confidence = float(conf)
            class_name = model.names[class_id]

            print(
                f"{class_name} | "
                f"confiance = {confidence:.3f}"
            )

    # sauvegarder la frame annotée
    annotated = result.plot()

    output_path = PROJECT_DIR / f"real_frame_{i+1}.jpg"
    cv2.imwrite(str(output_path), annotated)

    print("Image :", output_path)

cap.release()

In [ ]:
import cv2
from pathlib import Path
from ultralytics import YOLO
from collections import deque

# =========================
# PATHS
# =========================

PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")

MODEL_PATH = (
    PROJECT_DIR
    / "runs"
    / "safe_driving_v3"
    / "weights"
    / "best.pt"
)

VIDEO_PATH = PROJECT_DIR / "video_test.mp4"
OUTPUT_PATH = PROJECT_DIR / "safe_driving_v3_stable.mp4"

# =========================
# MODEL
# =========================

model = YOLO(str(MODEL_PATH))

# =========================
# PARAMETERS
# =========================

YOLO_CONFIDENCE = 0.20

SEATBELT_THRESHOLD = 0.25
NO_SEATBELT_THRESHOLD = 0.30
MOBILE_THRESHOLD = 0.30

# Temporal smoothing
WINDOW_SIZE = 15

# Number of positive frames needed
MIN_POSITIVE_FRAMES = 9

# Number of frames required to change state
CHANGE_CONFIRMATION = 5

# =========================
# VIDEO
# =========================

cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    raise Exception("Impossible d'ouvrir la vidéo.")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    str(OUTPUT_PATH),
    fourcc,
    fps,
    (width, height)
)

# =========================
# HISTORIES
# =========================

seatbelt_history = deque(maxlen=WINDOW_SIZE)
no_seatbelt_history = deque(maxlen=WINDOW_SIZE)

# =========================
# STABLE STATE
# =========================

current_status = "CHECKING SEATBELT"

candidate_status = None
candidate_count = 0

frame_number = 0

# =========================
# MAIN LOOP
# =========================

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1

    # =========================
    # YOLO
    # =========================

    results = model.predict(
        source=frame,
        conf=YOLO_CONFIDENCE,
        verbose=False
    )

    result = results[0]

    # =========================
    # CURRENT DETECTIONS
    # =========================

    seatbelt_detected = False
    no_seatbelt_detected = False
    mobile_detected = False

    for cls, conf in zip(
        result.boxes.cls,
        result.boxes.conf
    ):

        class_id = int(cls)
        confidence = float(conf)

        class_name = model.names[class_id]

        if (
            class_name == "seatbelt"
            and confidence >= SEATBELT_THRESHOLD
        ):
            seatbelt_detected = True

        elif (
            class_name == "no_seatbelt"
            and confidence >= NO_SEATBELT_THRESHOLD
        ):
            no_seatbelt_detected = True

        elif (
            class_name == "mobile"
            and confidence >= MOBILE_THRESHOLD
        ):
            mobile_detected = True

    # =========================
    # UPDATE HISTORY
    # =========================

    seatbelt_history.append(seatbelt_detected)
    no_seatbelt_history.append(no_seatbelt_detected)

    seatbelt_positive = sum(seatbelt_history)
    no_seatbelt_positive = sum(no_seatbelt_history)

    # =========================
    # TEMPORAL CONFIRMATION
    # =========================

    seatbelt_confirmed = (
        seatbelt_positive >= MIN_POSITIVE_FRAMES
    )

    no_seatbelt_confirmed = (
        no_seatbelt_positive >= MIN_POSITIVE_FRAMES
    )

    # =========================
    # DETERMINE NEW CANDIDATE
    # =========================

    if mobile_detected:

        proposed_status = "WARNING - MOBILE DETECTED"

    elif (
        no_seatbelt_confirmed
        and not seatbelt_confirmed
    ):

        proposed_status = "WARNING - NO SEATBELT"

    elif seatbelt_confirmed:

        proposed_status = "SAFE - SEATBELT CONFIRMED"

    elif no_seatbelt_confirmed:

        proposed_status = "WARNING - NO SEATBELT"

    else:

        proposed_status = "CHECKING SEATBELT"

    # =========================
    # STABILITY LOGIC
    # =========================

    if proposed_status != current_status:

        if proposed_status == candidate_status:

            candidate_count += 1

        else:

            candidate_status = proposed_status
            candidate_count = 1

        # Change status only after several
        # consecutive confirmations
        if candidate_count >= CHANGE_CONFIRMATION:

            current_status = proposed_status

            candidate_status = None
            candidate_count = 0

    else:

        candidate_status = None
        candidate_count = 0

    # =========================
    # DRAW
    # =========================

    annotated_frame = result.plot()

    # Status
    cv2.putText(
        annotated_frame,
        current_status,
        (20, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 255, 255),
        2
    )

    # Seatbelt history
    cv2.putText(
        annotated_frame,
        f"Seatbelt: {seatbelt_positive}/{len(seatbelt_history)}",
        (20, 90),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    # No seatbelt history
    cv2.putText(
        annotated_frame,
        f"No seatbelt: {no_seatbelt_positive}/{len(no_seatbelt_history)}",
        (20, 125),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    # Candidate
    cv2.putText(
        annotated_frame,
        f"Candidate: {candidate_status} ({candidate_count}/{CHANGE_CONFIRMATION})",
        (20, 160),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        2
    )

    # Frame
    cv2.putText(
        annotated_frame,
        f"Frame: {frame_number}/{total_frames}",
        (20, 195),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    # =========================
    # WRITE
    # =========================

    out.write(annotated_frame)

# =========================
# RELEASE
# =========================

cap.release()
out.release()

print("Video terminée.")
print("Résultat :", OUTPUT_PATH)